In [2]:
import os
import requests

datasets = [
    "title.basics.tsv.gz",   # Titles, years, genres
    "title.ratings.tsv.gz",  # Ratings and vote counts
    "title.crew.tsv.gz",     # Directors and writers
    "name.basics.tsv.gz",    # Actor/Director names
    "title.akas.tsv.gz"      # Regional titles/translations
]

base_url = "https://datasets.imdbws.com/"
save_dir = "./imdb_raw_data"

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

def download_files():
    for file in datasets:
        print(f"Downloading {file}...")
        response = requests.get(base_url + file, stream=True)
        with open(os.path.join(save_dir, file), "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
        print(f"Finished {file}")

download_files()

Finished title.basics.tsv.gz
Finished title.ratings.tsv.gz
Finished title.crew.tsv.gz
Finished name.basics.tsv.gz
Finished title.akas.tsv.gz


In [17]:
import pandas as pd

SAMPLE_SIZE = 100000 

def load_local_data():
    print("Loading datasets...")
    
    params = {
        'sep': '\t',
        'compression': 'gzip',
        'nrows': SAMPLE_SIZE,
        'low_memory': False,
        'na_values': '\\N'
    }

    # Loading the core files
    df_basics = pd.read_csv(r'imdb_raw_data\title.basics.tsv.gz', **params)
    df_ratings = pd.read_csv(r'imdb_raw_data\title.ratings.tsv.gz', **params)
    df_crew = pd.read_csv(r'imdb_raw_data\title.crew.tsv.gz', **params)
    df_names = pd.read_csv(r'imdb_raw_data\name.basics.tsv.gz', **params)
    df_akas = pd.read_csv(r'imdb_raw_data\title.akas.tsv.gz', **params)

    print("Success! Data loaded into DataFrames.")
    return df_basics, df_ratings, df_crew, df_names, df_akas

# Execute load
df_basics, df_ratings, df_crew, df_names, df_akas = load_local_data()

Loading datasets...
Success! Data loaded into DataFrames.


In [18]:
print("="*20 + "DF Basics" + "="*20)
print(df_basics.head())
print("="*20 + "DF Ratings" + "="*20)
print(df_ratings.head())
print("="*20 + "DF Crew" + "="*20)
print(df_crew.head())
print("="*20 + "DF Names" + "="*20)
print(df_names.head())
print("="*20 + "DF Akas" + "="*20)
print(df_akas.head())

====================DF Basics====================
      tconst titleType            primaryTitle           originalTitle  \
0  tt0000001     short              Carmencita              Carmencita   
1  tt0000002     short  Le clown et ses chiens  Le clown et ses chiens   
2  tt0000003     short            Poor Pierrot          Pauvre Pierrot   
3  tt0000004     short             Un bon bock             Un bon bock   
4  tt0000005     short        Blacksmith Scene        Blacksmith Scene   

   isAdult  startYear  endYear  runtimeMinutes                    genres  
0        0     1894.0      NaN             1.0         Documentary,Short  
1        0     1892.0      NaN             5.0           Animation,Short  
2        0     1892.0      NaN             5.0  Animation,Comedy,Romance  
3        0     1892.0      NaN            12.0           Animation,Short  
4        0     1893.0      NaN             1.0                     Short  
====================DF Ratings====================
    

In [19]:
print("="*20 + "DF Basics Info" + "="*20)
print(df_basics.info())
print("="*20 + "DF Ratings Info" + "="*20)
print(df_ratings.info())
print("="*20 + "DF Crew Info" + "="*20)
print(df_crew.info())
print("="*20 + "DF Names Info" + "="*20)
print(df_names.info())
print("="*20 + "DF Akas Info" + "="*20)
print(df_akas.info())

====================DF Basics Info====================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   tconst          100000 non-null  object 
 1   titleType       100000 non-null  object 
 2   primaryTitle    100000 non-null  object 
 3   originalTitle   100000 non-null  object 
 4   isAdult         100000 non-null  int64  
 5   startYear       99986 non-null   float64
 6   endYear         4168 non-null    float64
 7   runtimeMinutes  87504 non-null   float64
 8   genres          94422 non-null   object 
dtypes: float64(3), int64(1), object(5)
memory usage: 6.9+ MB
None
====================DF Ratings Info====================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   tco

In [20]:
df_basics = df_basics.drop(columns=['isAdult']).copy()

df_basics['startYear'] = pd.to_numeric(df_basics['startYear'], errors='coerce').astype('Int64')
df_basics['endYear'] = pd.to_numeric(df_basics['endYear'], errors='coerce').astype('Int64')

df_basics['runtimeMinutes'] = pd.to_numeric(df_basics['runtimeMinutes'], errors='coerce').astype('Int64')

string_cols = ['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'genres']
for col in string_cols:
    if col == 'genres':
        df_basics[col] = df_basics[col].fillna('Unknown')

    df_basics[col] = df_basics[col].astype('string')

print(df_basics.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   tconst          100000 non-null  string
 1   titleType       100000 non-null  string
 2   primaryTitle    100000 non-null  string
 3   originalTitle   100000 non-null  string
 4   startYear       99986 non-null   Int64 
 5   endYear         4168 non-null    Int64 
 6   runtimeMinutes  87504 non-null   Int64 
 7   genres          100000 non-null  string
dtypes: Int64(3), string(5)
memory usage: 6.4 MB
None


In [22]:
df_ratings['tconst'] = df_ratings['tconst'].astype('string')
df_ratings['averageRating'] = df_ratings['averageRating'].astype('float64')
df_ratings['numVotes'] = df_ratings['numVotes'].astype('Int64')

print("--- Null Values Count ---")
print(df_ratings.isnull().sum())

print("--- Info ---")
print(df_ratings.info())

--- Null Values Count ---
tconst           0
averageRating    0
numVotes         0
dtype: int64
--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   tconst         100000 non-null  string 
 1   averageRating  100000 non-null  float64
 2   numVotes       100000 non-null  Int64  
dtypes: Int64(1), float64(1), string(1)
memory usage: 2.4 MB
None


In [24]:
df_crew['tconst'] = df_crew['tconst'].astype('string')

df_crew['directors'] = df_crew['directors'].astype('string')
df_crew['writers'] = df_crew['writers'].astype('string')

print(df_crew.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   tconst     100000 non-null  string
 1   directors  97631 non-null   string
 2   writers    89743 non-null   string
dtypes: string(3)
memory usage: 2.3 MB
None


In [ ]:
df_names['birthYear'] = pd.to_numeric(df_names['birthYear'], errors='coerce').astype('Int64')
df_names['deathYear'] = pd.to_numeric(df_names['deathYear'], errors='coerce').astype('Int64')

df_names['nconst'] = df_names['nconst'].astype('string')
df_names['primaryName'] = df_names['primaryName'].astype('string')

bridge_professions = df_names[['nconst', 'primaryProfession']].dropna().copy()

bridge_professions = bridge_professions['primaryProfession'].str.split(',')
bridge_professions = bridge_professions.explode('primaryProfession')

bridge_professions.columns = ['nconst', 'profession']
bridge_professions[['nconst', 'profession']] = bridge_professions[['nconst', 'profession']].astype('string')

bridge_known_for = df_names[['nconst', 'knownForTitles']].dropna().copy()

bridge_known_for['knownForTitles'] = 